# Education 360° — Data Inspection
**Objective:** Load all source CSVs and understand their structure, size, columns, types, and sample records.

Run from the project folder or the `Python` folder. The notebook expects CSVs in `../Dataset/`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)

PROJECT_PATH = Path.cwd().parent if Path.cwd().name.lower() == "python" else Path.cwd()
DATA_PATH = PROJECT_PATH / "Dataset"

# Accommodate the source filename spelling found in the project.
FILE_CANDIDATES = {
    "admissions": ["admissions.csv"],
    "branches dim": ["branches_dim.csv", "braches_dim.csv"],
    "calls": ["calls.csv"],
    "courses dim": ["courses_dim.csv"],
    "date table": ["DateTable.csv", "date_table.csv"],
    "employees": ["employees.csv"],
    "fees": ["fees.csv"],
    "leads": ["leads.csv"],
    "marketing": ["marketing.csv"],
    "placements": ["placements.csv"],
    "students dim": ["students_dim.csv"]
}

def load_table(table_name):
    candidates = FILE_CANDIDATES[table_name]
    for filename in candidates:
        path = DATA_PATH / filename
        if path.exists():
            return pd.read_csv(path)
    raise FileNotFoundError(
        f"Could not find {table_name}. Tried: {candidates}. "
        f"Expected dataset folder: {DATA_PATH}"
    )

datasets = {}
for table in FILE_CANDIDATES:
    try:
        datasets[table] = load_table(table)
        print(f"{table:15} {datasets[table].shape}")
    except FileNotFoundError as exc:
        print(exc)

## 1. Dataset overview

In [ ]:
overview = pd.DataFrame([
    {
        "Table": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Values": int(df.isna().sum().sum()),
        "Duplicate Rows": int(df.duplicated().sum()),
        "Memory (KB)": round(df.memory_usage(deep=True).sum() / 1024, 2)
    }
    for name, df in datasets.items()
])
display(overview)

## 2. Column names and data types

In [ ]:
for name, df in datasets.items():
    print(f"\\n{'='*65}\\n{name.upper()}\\n{'='*65}")
    display(pd.DataFrame({
        "Column": df.columns,
        "Data Type": [str(x) for x in df.dtypes],
        "Missing": df.isna().sum().values,
        "Unique Values": df.nunique(dropna=True).values
    }))

## 3. Preview each table

In [ ]:
for name, df in datasets.items():
    print(f"\\n{name.upper()} — first five rows")
    display(df.head())

## 4. Numeric summary

In [ ]:
for name, df in datasets.items():
    numeric = df.select_dtypes(include="number")
    if not numeric.empty:
        print(f"\\n{name.upper()}")
        display(numeric.describe().T)

## Findings to record
Summarize table sizes, unusual columns, date fields, numeric fields, and any filename or schema inconsistencies.